# Лабораторная работа №6 - Ансамбли моделей машинного обучения. Часть 2.

### Цель лабораторной работы: изучение ансамблей моделей машинного обучения.

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

from sklearn.ensemble import StackingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

import warnings
warnings.filterwarnings('ignore')

In [9]:
df = pd.read_csv('loan_approval_dataset.csv')
df.columns = df.columns.str.strip()
print(df.shape)

(4269, 13)


In [8]:
df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [10]:
print("Пропуски:\n", df.isnull().sum())
print("\nТипы данных:\n", df.dtypes)

Пропуски:
 loan_id                     0
no_of_dependents            0
education                   0
self_employed               0
income_annum                0
loan_amount                 0
loan_term                   0
cibil_score                 0
residential_assets_value    0
commercial_assets_value     0
luxury_assets_value         0
bank_asset_value            0
loan_status                 0
dtype: int64

Типы данных:
 loan_id                     int64
no_of_dependents            int64
education                     str
self_employed                 str
income_annum                int64
loan_amount                 int64
loan_term                   int64
cibil_score                 int64
residential_assets_value    int64
commercial_assets_value     int64
luxury_assets_value         int64
bank_asset_value            int64
loan_status                   str
dtype: object


In [11]:
# Удаляем ненужный столбец
df = df.drop(columns=['loan_id'])

# Кодируем категориальные признаки
le = LabelEncoder()
df['education'] = le.fit_transform(df['education'])        # Graduate/Not Graduate → 1/0
df['self_employed'] = le.fit_transform(df['self_employed']) # Yes/No → 1/0
df['loan_status'] = le.fit_transform(df['loan_status'])     # Approved/Rejected → 0/1

print("Данные после предобработки:")
df.head()

Данные после предобработки:


,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,2,0,0,9600000,29900000,12,778,2400000,17600000,22700000,8000000,0
1,0,1,1,4100000,12200000,8,417,2700000,2200000,8800000,3300000,1
2,3,0,0,9100000,29700000,20,506,7100000,4500000,33300000,12800000,1
3,3,0,0,8200000,30700000,8,467,18200000,3300000,23300000,7900000,1
4,5,1,1,9800000,24200000,20,382,12400000,8200000,29400000,5000000,1


In [12]:
X = df.drop(columns=['loan_status'])
y = df['loan_status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Обучающая выборка:", X_train_scaled.shape)
print("Тестовая выборка:", X_test_scaled.shape)

Обучающая выборка: (3415, 11)
Тестовая выборка: (854, 11)


In [13]:
estimators = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('gbm', GradientBoostingClassifier(n_estimators=50, random_state=42))
]

stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression()
)

stacking_model.fit(X_train_scaled, y_train)
y_pred_stacking = stacking_model.predict(X_test_scaled)

print("=== Стекинг ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_stacking):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_stacking):.4f}")

=== Стекинг ===
Accuracy: 0.9789
F1-score: 0.9714


In [14]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    max_iter=500,
    random_state=42
)

mlp_model.fit(X_train_scaled, y_train)
y_pred_mlp = mlp_model.predict(X_test_scaled)

print("=== Многослойный перцептрон (MLP) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_mlp):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_mlp):.4f}")

=== Многослойный перцептрон (MLP) ===
Accuracy: 0.9649
F1-score: 0.9530


In [15]:
results = pd.DataFrame({
    'Модель': ['Стекинг', 'MLP (перцептрон)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_stacking),
        accuracy_score(y_test, y_pred_mlp)
    ],
    'F1-score': [
        f1_score(y_test, y_pred_stacking),
        f1_score(y_test, y_pred_mlp)
    ]
})

results = results.sort_values('Accuracy', ascending=False).reset_index(drop=True)
results

,Модель,Accuracy,F1-score
0,Стекинг,0.978923,0.971429
1,MLP (перцептрон),0.964871,0.952978
